In [2]:
import pandas as pd

df = pd.read_csv("student_performance.csv")

print("Dataset Shape:", df.shape)
print("\nFirst 5 Rows:")
display(df.head())

print("\nColumn Names:")
print(df.columns.tolist())

print("\nData Types:")
print(df.dtypes)

print("\nMissing Values:")
print(df.isnull().sum())

Dataset Shape: (24, 6)

First 5 Rows:


,Student_ID,Age,Study_Hours,Attendance,Department,Result
0,1,20.0,5.0,85.0,CSE,Pass
1,2,21.0,2.0,60.0,ECE,Fail
2,3,20.0,6.0,90.0,IT,Pass
3,4,22.0,1.0,45.0,CSE,Fail
4,5,21.0,4.0,80.0,ECE,Pass



Column Names:
['Student_ID', 'Age', 'Study_Hours', 'Attendance', 'Department', 'Result']

Data Types:
Student_ID       int64
Age            float64
Study_Hours    float64
Attendance     float64
Department         str
Result             str
dtype: object

Missing Values:
Student_ID     0
Age            1
Study_Hours    1
Attendance     1
Department     1
Result         0
dtype: int64


In [3]:
# Check target distribution

print("Result Distribution:")
print(df["Result"].value_counts())

print("\nResult Percentage:")
print(df["Result"].value_counts(normalize=True) * 100)

# Check duplicate records
print("\nDuplicate Rows:", df.duplicated().sum())

# Check missing values again
print("\nMissing Values:")
print(df.isnull().sum())

Result Distribution:
Result
Pass    12
Fail    12
Name: count, dtype: int64

Result Percentage:
Result
Pass    50.0
Fail    50.0
Name: proportion, dtype: float64

Duplicate Rows: 0

Missing Values:
Student_ID     0
Age            1
Study_Hours    1
Attendance     1
Department     1
Result         0
dtype: int64


In [4]:
# Import required libraries

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# Separate features and target

X = df.drop(columns=["Student_ID", "Result"])
y = df["Result"]

print("Features:")
print(X.head())

print("\nTarget:")
print(y.head())

print("\nFeature Shape:", X.shape)
print("Target Shape:", y.shape)

Features:
    Age  Study_Hours  Attendance Department
0  20.0          5.0        85.0        CSE
1  21.0          2.0        60.0        ECE
2  20.0          6.0        90.0         IT
3  22.0          1.0        45.0        CSE
4  21.0          4.0        80.0        ECE

Target:
0    Pass
1    Fail
2    Pass
3    Fail
4    Pass
Name: Result, dtype: str

Feature Shape: (24, 4)
Target Shape: (24,)


In [5]:
# Split dataset into training and testing sets

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Display dataset sizes

print("Training Features:", X_train.shape)
print("Testing Features:", X_test.shape)

print("\nTraining Target Distribution:")
print(y_train.value_counts())

print("\nTesting Target Distribution:")
print(y_test.value_counts())

Training Features: (19, 4)
Testing Features: (5, 4)

Training Target Distribution:
Result
Pass    10
Fail     9
Name: count, dtype: int64

Testing Target Distribution:
Result
Fail    3
Pass    2
Name: count, dtype: int64


In [6]:
# Define numerical and categorical columns

numeric_features = ["Age", "Study_Hours", "Attendance"]
categorical_features = ["Department"]


# Preprocessing for Logistic Regression

logistic_preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), numeric_features),

        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore"))
        ]), categorical_features)
    ]
)


# Preprocessing for Decision Tree

tree_preprocessor = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), numeric_features),

        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore"))
        ]), categorical_features)
    ]
)

print("Both preprocessing pipelines created successfully!")

Both preprocessing pipelines created successfully!


In [7]:
# Create Logistic Regression pipeline

logistic_model = Pipeline([
    ("preprocessor", logistic_preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])


# Create Decision Tree pipeline

tree_model = Pipeline([
    ("preprocessor", tree_preprocessor),
    ("classifier", DecisionTreeClassifier(
        max_depth=3,
        random_state=42
    ))
])


# Train both models

logistic_model.fit(X_train, y_train)
tree_model.fit(X_train, y_train)

print("Logistic Regression model trained successfully!")
print("Decision Tree model trained successfully!")

Logistic Regression model trained successfully!
Decision Tree model trained successfully!


In [8]:
# Generate predictions

logistic_predictions = logistic_model.predict(X_test)
tree_predictions = tree_model.predict(X_test)


# Function to calculate evaluation metrics

def evaluate_model(y_true, y_pred, model_name):

    return {
        "Model": model_name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(
            y_true, y_pred, pos_label="Pass", zero_division=0
        ),
        "Recall": recall_score(
            y_true, y_pred, pos_label="Pass", zero_division=0
        ),
        "F1-Score": f1_score(
            y_true, y_pred, pos_label="Pass", zero_division=0
        )
    }


# Evaluate both models

logistic_results = evaluate_model(
    y_test, logistic_predictions, "Logistic Regression"
)

tree_results = evaluate_model(
    y_test, tree_predictions, "Decision Tree"
)


# Display comparison table

comparison_df = pd.DataFrame([
    logistic_results,
    tree_results
])

print("MODEL COMPARISON")
display(comparison_df.round(3))

MODEL COMPARISON


,Model,Accuracy,Precision,Recall,F1-Score
0,Logistic Regression,1.0,1.0,1.0,1.0
1,Decision Tree,1.0,1.0,1.0,1.0


In [9]:
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import make_scorer, f1_score

# Create 5-fold cross-validation

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Define evaluation metrics

scoring = {
    "accuracy": "accuracy",
    "f1_score": make_scorer(f1_score, pos_label="Pass")
}

# Evaluate Logistic Regression

logistic_cv = cross_validate(
    logistic_model,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    return_train_score=True
)

# Evaluate Decision Tree

tree_cv = cross_validate(
    tree_model,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    return_train_score=True
)

# Display cross-validation results

cv_results = pd.DataFrame([
    {
        "Model": "Logistic Regression",
        "Mean Training Accuracy": logistic_cv["train_accuracy"].mean(),
        "Mean CV Accuracy": logistic_cv["test_accuracy"].mean(),
        "Mean CV F1": logistic_cv["test_f1_score"].mean()
    },
    {
        "Model": "Decision Tree",
        "Mean Training Accuracy": tree_cv["train_accuracy"].mean(),
        "Mean CV Accuracy": tree_cv["test_accuracy"].mean(),
        "Mean CV F1": tree_cv["test_f1_score"].mean()
    }
])

display(cv_results.round(3))

,Model,Mean Training Accuracy,Mean CV Accuracy,Mean CV F1
0,Logistic Regression,1.0,1.00,1.00
1,Decision Tree,1.0,0.95,0.96


In [10]:
# Compare training and cross-validation performance

overfitting_analysis = pd.DataFrame([
    {
        "Model": "Logistic Regression",
        "Training Accuracy": logistic_cv["train_accuracy"].mean(),
        "Validation Accuracy": logistic_cv["test_accuracy"].mean()
    },
    {
        "Model": "Decision Tree",
        "Training Accuracy": tree_cv["train_accuracy"].mean(),
        "Validation Accuracy": tree_cv["test_accuracy"].mean()
    }
])

overfitting_analysis["Performance Gap"] = (
    overfitting_analysis["Training Accuracy"] -
    overfitting_analysis["Validation Accuracy"]
)

display(overfitting_analysis.round(3))

print("Decision Tree Depth:", 
      tree_model.named_steps["classifier"].get_depth())

print("Decision Tree Leaves:", 
      tree_model.named_steps["classifier"].get_n_leaves())

,Model,Training Accuracy,Validation Accuracy,Performance Gap
0,Logistic Regression,1.0,1.00,0.00
1,Decision Tree,1.0,0.95,0.05


Decision Tree Depth: 1
Decision Tree Leaves: 2


In [12]:
# Combine all evaluation results

final_comparison = comparison_df.merge(
    cv_results,
    on="Model"
)

final_comparison = final_comparison.merge(
    overfitting_analysis[["Model", "Performance Gap"]],
    on="Model"
)

# Add Decision Tree complexity information

final_comparison["Tree Depth"] = [
    None,
    tree_model.named_steps["classifier"].get_depth()
]

final_comparison["Tree Leaves"] = [
    None,
    tree_model.named_steps["classifier"].get_n_leaves()
]

# Save comparison table

final_comparison.to_csv("model_comparison.csv", index=False)

print("FINAL MODEL COMPARISON")
display(final_comparison.round(3))

print("\nModel comparison saved successfully!")

FINAL MODEL COMPARISON


,Model,Accuracy,Precision,Recall,F1-Score,Mean Training Accuracy,Mean CV Accuracy,Mean CV F1,Performance Gap,Tree Depth,Tree Leaves
0,Logistic Regression,1.0,1.0,1.0,1.0,1.0,1.00,1.00,0.00,NaN,NaN
1,Decision Tree,1.0,1.0,1.0,1.0,1.0,0.95,0.96,0.05,1.0,2.0



Model comparison saved successfully!
